# SupportBot - data exploration

Exploratory data analysis of the **Bitext customer-support dataset** (training data) and the
**CLINC150 out-of-scope queries** (used only to test out-of-scope detection).

Run `python src/data_download.py` first. Then *Run All*.

In [1]:
import sys, re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from common import tokenize
import plots  # shared plot style

DATA = ROOT / "data"
raw = pd.read_csv(DATA / "bitext_raw.csv")
print(raw.shape)
raw.head()

(26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


## 1. Columns and missing values

In [2]:
print(raw.dtypes, "\n")
print("missing values:\n", raw.isna().sum())
print("\nexact duplicate messages:", raw["instruction"].duplicated().sum())

flags          str
instruction    str
category       str
intent         str
response       str
dtype: object 

missing values:
 flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

exact duplicate messages: 2237


## 2. Intents and categories
Is the dataset balanced? (Balanced classes mean accuracy and macro-F1 will be close.)

In [3]:
print(raw["category"].nunique(), "categories,", raw["intent"].nunique(), "intents")
counts = raw["intent"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(counts.index, counts.values, color=plots.SERIES[0], height=0.7)
ax.set(title="Messages per intent", xlabel="number of messages")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
print("smallest / largest intent:", counts.min(), "/", counts.max())

11 categories, 27 intents
smallest / largest intent: 950 / 1000


C:\Users\Hp\AppData\Local\Temp\ipykernel_11240\1252449758.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
raw.groupby("category")["intent"].unique().apply(lambda x: ", ".join(sorted(x))).to_frame("intents")

,intents
category,
ACCOUNT,"create_account, delete_account, edit_account, ..."
CANCEL,check_cancellation_fee
CONTACT,"contact_customer_service, contact_human_agent"
DELIVERY,"delivery_options, delivery_period"
FEEDBACK,"complaint, review"
INVOICE,"check_invoice, get_invoice"
ORDER,"cancel_order, change_order, place_order, track..."
PAYMENT,"check_payment_methods, payment_issue"
REFUND,"check_refund_policy, get_refund, track_refund"


## 3. Message length
The from-scratch Transformer uses `max_len = 32` tokens - check this covers almost all messages.

In [5]:
raw["n_tokens"] = raw["instruction"].apply(lambda t: len(tokenize(t)))
print(raw["n_tokens"].describe())
print("share of messages with <= 32 tokens:", (raw["n_tokens"] <= 32).mean().round(4))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(raw["n_tokens"], bins=40, color=plots.SERIES[0])
ax.axvline(32, color=plots.INK, ls="--", lw=1.5)
ax.set(title="Tokens per customer message", xlabel="tokens", ylabel="messages")
plt.tight_layout(); plt.show()

count    26872.000000
mean        10.073906
std          3.304249
min          1.000000
25%          8.000000
50%         10.000000
75%         12.000000
max         26.000000
Name: n_tokens, dtype: float64
share of messages with <= 32 tokens: 1.0


C:\Users\Hp\AppData\Local\Temp\ipykernel_11240\1505403271.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Example messages per intent

In [6]:
raw.groupby("intent").head(2)[["intent", "instruction"]].sort_values("intent").head(20)

,intent,instruction
0,cancel_order,question about cancelling order {{Order Number}}
1,cancel_order,i have a question about cancelling oorder {{Or...
998,change_order,I try to change several bloody items of order ...
999,change_order,modifying purchase {{Order Number}}
1995,change_shipping_address,give me information about a delivery address m...
1996,change_shipping_address,need to update my address
2968,check_cancellation_fee,"I can't ifnd the bloody termination charge, I ..."
2969,check_cancellation_fee,what is the fee for canceling the contract?
3918,check_invoice,show me invoice{{Invoice Number}}
3919,check_invoice,need to see my invoicesfrom {{Person Name}}


## 5. Linguistic variation flags

Bitext tags each message with flags describing its style, e.g. **B** basic, **L** colloquial,
**Q** question, **P** polite, **Z** typos/errors, **M** morphological variation, **W** offensive.

In [7]:
flag_counts = pd.Series([f for fl in raw["flags"].astype(str) for f in fl]).value_counts()
flag_counts.to_frame("messages with this flag")

,messages with this flag
B,26872
L,24115
Q,8968
I,7839
Z,5286
M,4920
C,2646
K,2227
E,1882
P,1329


In [8]:
print("messages with typos (Z flag):")
raw[raw["flags"].astype(str).str.contains("Z")]["instruction"].head(8).tolist()

messages with typos (Z flag):


['i have a question about cancelling oorder {{Order Number}}',
 'i need help cancelling puchase {{Order Number}}',
 'help  me canceling purchase {{Order Number}}',
 'assistance with cancelling pucrhase {{Order Number}}',
 'help cnceling order {{Order Number}}',
 'assistnace cancelling order {{Order Number}}',
 'assistance with cancelling pucrhase {{Order Number}}',
 'want assistance to cancel order  {{Order Number}}']

## 6. Responses and placeholders
Replies contain slots such as `{{Order Number}}`. The app shows them as `[Order Number]`.

In [9]:
slots = raw["response"].str.findall(r"\{\{(.*?)\}\}").explode().value_counts()
print("different placeholders:", len(slots))
slots.head(15).to_frame("count")

different placeholders: 391


,count
response,
Order Number,5122
Account Type,4429
Account Category,3078
Online Order Interaction,2699
Customer Support Phone Number,2635
Website URL,2534
Customer Support Hours,2325
Invoice Number,1513
Online Company Portal Info,1055


In [10]:
raw["response_words"] = raw["response"].str.split().str.len()
raw["response_words"].describe()

count    26872.000000
mean       104.789037
std         52.966204
min          9.000000
25%         72.000000
50%         90.000000
75%        124.000000
max        402.000000
Name: response_words, dtype: float64

## 7. Vocabulary

In [11]:
from collections import Counter
vocab = Counter(tok for t in raw["instruction"] for tok in tokenize(t))
print("vocabulary size:", len(vocab))
print("words that appear once:", sum(1 for c in vocab.values() if c == 1))
vocab.most_common(25)

vocabulary size: 2890
words that appear once: 1670


[('i', 18875),
 ('to', 16400),
 ('{', 14084),
 ('}', 14084),
 ('the', 6413),
 ('help', 6319),
 ('a', 6086),
 ('account', 5400),
 ('do', 5348),
 ('can', 5274),
 ('?', 5072),
 ('order', 4833),
 ('my', 4774),
 ('how', 4207),
 ('need', 4059),
 ('of', 3994),
 ('me', 3661),
 ('what', 3136),
 ('want', 3110),
 ('assistance', 3005),
 ('have', 2957),
 ('number', 2953),
 ('know', 2691),
 ('with', 2553),
 ('see', 2545)]

## 8. Train / validation / test split (stratified, seed 42)

In [12]:
splits = {n: pd.read_csv(DATA / f"{n}.csv") for n in ("train", "val", "test")}
pd.DataFrame({n: [len(d), d["intent"].nunique()] for n, d in splits.items()}, index=["rows", "intents"])

,train,val,test
rows,19708,2463,2464
intents,27,27,27


In [13]:
# overlap check: the same message must not be in train and test
print("test messages also in train:", splits["test"]["text"].isin(splits["train"]["text"]).sum())

test messages also in train: 0


## 9. Out-of-scope queries (CLINC150)

These are *not* customer-support requests. The model never trains on them; they are used only to
measure whether SupportBot can say "this is not something I handle".

In [14]:
oos_val, oos_test = pd.read_csv(DATA / "oos_val.csv"), pd.read_csv(DATA / "oos_test.csv")
print(len(oos_val), "validation /", len(oos_test), "test")
oos_test.sample(10, random_state=0)

100 validation / 1000 test


,text
993,can you detect where i placed my wallet
859,how do you relieve a hangover
298,what year did my grandparents get married
553,did a man really die trying to be loved by a h...
672,tell me where i can get the cheapest tennis ba...
971,can you find my car in the parking lot
27,wash windshield
231,what is the median salary for someone with an ...
306,who played ronald weasly
706,search up the cast in the walking dead


## 10. Observations

* 27 intents in 11 categories, roughly balanced → accuracy and macro-F1 behave similarly.
* Messages are short; 32 tokens covers almost every message.
* Many paraphrases of the same request → in-distribution accuracy will be very high, so the
  more informative tests are **typos (noisy test set)** and **out-of-scope detection**.
* Replies contain placeholders that a real system fills from the order database.